# Transformación de datos: esperanza de vida

**Estudiante:** Jorge Adalberto Zuñiga Oliva  
**Matrícula:** 82005 · **Grupo:** 512  
**Materia:** Tópicos de Big Data

In [1]:
from pathlib import Path
import sys
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/curso.py").exists())
sys.path.insert(0, str(RAIZ / "scripts"))
from curso import (leer_owid, metadatos, indicador, solo_paises, guardar_figura,
                   guardar_tabla, U1, U3, SALIDA, FUENTES)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
pd.set_option("display.max_rows", 12)
pd.set_option("display.max_columns", 12)

## 1. Fuente y estructura
Se utiliza la copia local del indicador de esperanza de vida de OWID. Los metadatos describen
esperanza de vida **de periodo**, en años; no es el promedio de edades de quienes murieron ese año.
La descarga, URL y SHA-256 se registran en `datos_fuente/manifest.json`.
[Fuente y definición](https://ourworldindata.org/grapher/life-expectancy).

In [2]:
df = leer_owid("life-expectancy")
display(df.head())
print("Dimensiones:", df.shape)
display(df.dtypes.to_frame("tipo"))
display(df.isna().sum().to_frame("faltantes"))

,entity,code,year,life_expectancy_0
0,Afghanistan,AFG,1950,28.1563
1,Afghanistan,AFG,1951,28.5836
2,Afghanistan,AFG,1952,29.0138
3,Afghanistan,AFG,1953,29.4521
4,Afghanistan,AFG,1954,29.6975


Dimensiones: (21565, 4)


,tipo
entity,object
code,object
year,int64
life_expectancy_0,float64


,faltantes
entity,0
code,1348
year,0
life_expectancy_0,0


## 2. Renombrar, indexar y filtrar
`rename` modifica etiquetas, no unidades. `set_index` devuelve otro DataFrame si no se asigna
el resultado. Se conserva el ejercicio de nombres que contienen la letra I.

In [3]:
datos = df.rename(columns={"entity": "Pais", "code": "Codigo", "year": "Anio",
                                    "life_expectancy_0": "Esperanza de Vida"})
por_pais = datos.set_index("Pais")
display(por_pais.loc[["Mexico"]].tail(3))
contiene_i = datos.loc[datos["Pais"].str.contains("i", case=False, regex=False)]
display(contiene_i.head())
print("Filas con I en el nombre:", len(contiene_i))

,Codigo,Anio,Esperanza de Vida
Pais,,,
Mexico,MEX,2021,69.7499
Mexico,MEX,2022,73.9731
Mexico,MEX,2023,75.0690


,Pais,Codigo,Anio,Esperanza de Vida
0,Afghanistan,AFG,1950,28.1563
1,Afghanistan,AFG,1951,28.5836
2,Afghanistan,AFG,1952,29.0138
3,Afghanistan,AFG,1953,29.4521
4,Afghanistan,AFG,1954,29.6975


Filas con I en el nombre: 13468


## 3. Faltantes y agregación
Los códigos ausentes suelen corresponder a entidades históricas o agregados; no son
automáticamente errores. Para la comparación se define el universo de códigos de tres letras,
excluyendo los agregados OWID. Se calcula un corte de 2020 y, por separado, el promedio histórico.
Promediar años de un país no equivale a calcular la esperanza de vida de su población.

In [4]:
display(datos.loc[datos["Codigo"].isna()].head(3))
paises = datos.loc[datos["Codigo"].astype("string").str.fullmatch(r"[A-Z]{3}", na=False)].copy()
paises = paises.dropna(subset=["Esperanza de Vida"])
corte = paises.loc[paises.Anio.eq(2020)].sort_values("Esperanza de Vida")
promedios = paises.groupby("Pais", as_index=False)["Esperanza de Vida"].mean()
display(corte[["Pais", "Esperanza de Vida"]].head(10))
display(promedios.head())
guardar_tabla(corte, "esperanza_vida_2020.csv", U1 / "data/processed")

,Pais,Codigo,Anio,Esperanza de Vida
375,Americas,NaN,1770,34.8
376,Americas,NaN,1830,34.8
377,Americas,NaN,1850,35.1


,Pais,Esperanza de Vida
3423,Central African Republic,50.5959
14077,Nigeria,53.0724
3497,Chad,53.0818
10536,Lesotho,55.1325
17852,Somalia,57.0948
18082,South Sudan,57.6479
11919,Mali,58.8577
7694,Guinea,59.3499
17331,Sierra Leone,59.6935
5735,Eswatini,59.8618


,Pais,Esperanza de Vida
0,Afghanistan,45.666349
1,Albania,68.773254
2,Algeria,58.263492
3,American Samoa,68.944123
4,Andorra,77.991046


CSV guardado: U1_1_probabilidad_estadistica/data/processed/esperanza_vida_2020.csv (236 filas)


## 4. Respuestas y conclusión
- **¿Qué cambia al renombrar?** Solo los nombres de columnas; los valores y su unidad permanecen.
- **¿Qué hacen `loc` y `set_index`?** `loc` selecciona por etiquetas o condiciones; `set_index`
  permite usar un país como etiqueta, que puede repetirse porque hay varios años.
- **¿Se deben borrar todos los nulos?** No. Se descartan solamente los que impiden la pregunta
  analizada; el criterio de códigos define el universo de esta comparación.
- **¿Qué significa el promedio?** El corte de 2020 compara entidades en el mismo año;
  el promedio histórico combina periodos de distinta duración y no debe usarse como ranking actual.

In [5]:
mx = corte.loc[corte.Pais.eq("Mexico")].iloc[0]
display(Markdown(f"En el corte de **2020**, México tiene **{mx['Esperanza de Vida']:.2f} años**. "
                 f"La tabla incluye **{len(corte)} entidades** con código de tres letras y dato disponible."))

En el corte de **2020**, México tiene **70.45 años**. La tabla incluye **236 entidades** con código de tres letras y dato disponible.